In [83]:
import pandas as pd
import numpy as np
import re
from datetime import datetime as dt
from datetime import timedelta, datetime


import random

In [84]:
#read in metadata
metadata_df = pd.read_csv("../mpox_build/phylogenetic/old_results/01-22-25/hmpxv1/metadata.tsv", sep ="\t")
metadata_df.head()

,accession,Unnamed: 0,genbank_accession_rev,strain,date,region,country,division,location,host,...,divergence,nonACGTN,QC_missing_data,QC_mixed_sites,QC_rare_mutations,QC_frame_shifts,QC_stop_codons,frame_shifts,is_reverse_complement,focus_areas
0,OQ339215,9113,OQ339215.1,MPXV/Germany/2022/ON/RKI696,2022-08-XX,Europe,Germany,NaN,NaN,Homo sapiens,...,69.0,0.0,good,good,good,good,good,NaN,False,Global
1,OQ469080,705,OQ469080.1,NY-NYCPHL-000316,2022-08-XX,North America,USA,New York,New York City,Homo sapiens,...,67.0,0.0,good,good,good,good,good,NaN,False,New York City
2,OQ469012,637,OQ469012.1,NY-NYCPHL-000008,2022-07-XX,North America,USA,New York,New York City,Homo sapiens,...,71.0,0.0,good,good,good,good,good,NaN,False,New York City
3,PQ374435,4726,PQ374435.1,hMpxV/USA/IL-RIPHL-050-0196/2024,2024-07-16,North America,USA,Illinois,Cook County IL,Homo sapiens,...,81.0,5.0,mediocre,good,good,good,good,NaN,False,Cook County
4,OP594192,7706,OP594192.1,MPXV/Germany/2022/RKI428,2022-09-XX,Europe,Germany,NaN,NaN,Homo sapiens,...,72.0,0.0,good,good,good,good,good,NaN,False,Global


In [85]:
## now subset it to columns of interest
metadata_df = metadata_df[["strain", "date", "accession", "authors", "institution",  'focus_areas' ]]

## print out tsv
metadata_df.to_csv("../figures/final_acknowledgements_table_genbank.tsv", sep = "\t")

In [86]:
## this does the same but only for the seqs in LAC
lac_df = metadata_df[metadata_df.focus_areas == "Los Angeles County"]
lac_df.to_csv("../figures/la_county_seqs.tsv", sep = "\t")

In [87]:
new_la_df = pd.read_csv('/Users/mparedes/Downloads/LACPHL_MPXV_Accessions.txt', sep="\t")



In [88]:
new_la_df = new_la_df[["GenBank"]]
new_la_df = new_la_df.rename(columns = {"GenBank": "accession"})
new_la_df

,accession
0,OP440572
1,OP440571
2,OP440570
3,OP440569
4,OP440568
...,...
494,PQ602736
495,PQ810947
496,PQ810946
497,PQ857767


In [89]:
lac_df_strains =lac_df[["accession"]].reset_index(drop = True)

In [44]:
lac_df_strains

,accession
0,OR643700
1,OP890548
2,OP440531
3,OP890575
4,OQ331002
...,...
714,OQ503814
715,OP539899
716,PP338782
717,OP890540


In [47]:
merged = lac_df_strains.merge(new_la_df, on='accession', how='outer', indicator=True)
merged

,accession,_merge
0,OR643700,both
1,OP890548,both
2,OP440531,left_only
3,OP890575,left_only
4,OQ331002,both
...,...,...
718,PP338782,both
719,OP890540,both
720,PP208824,both
721,PV173701,right_only


In [75]:
merged[merged["_merge"] == "right_only"]

,accession,_merge
721,PV173701,right_only
722,OQ503825,right_only


In [77]:
removed_df = merged.accession[merged["_merge"] == "left_only"].reset_index(drop = "true")

In [82]:
removed_df.to_csv("to_remove_mpox.tsv", sep = "\t")

In [79]:
removed_meta = lac_df.merge(removed_df, on= "accession")

In [81]:
removed_meta

,strain,date,accession,authors,institution,focus_areas
0,hMPX/Human/USA/CA-LACPHL-MA00049/2022,2022-07-20,OP440531,Hemarajata et al.,"Los Angeles County Public Health Laboratories,...",Los Angeles County
1,MPXV/Human/USA/CA-LACPHL-MA00318/2022,2022-11-09,OP890575,Hemarajata et al.,"Los Angeles County Public Health Laboratories,...",Los Angeles County
2,MPXV/Human/USA/CA-LACPHL-MA00505/2023,2023-04-12,OR499962,Garrigues et al.,Los Angeles County Public Health Laboratories,Los Angeles County
3,MPXV/Human/USA/CA-LACPHL-MA00439/2022,2022-10-09,OQ503835,Hemarajata et al.,"Los Angeles County Public Health Laboratories,...",Los Angeles County
4,MPXV/Human/USA/CA-LACPHL-MA00453/2022,2022-10-14,OQ503787,Hemarajata et al.,"Los Angeles County Public Health Laboratories,...",Los Angeles County
...,...,...,...,...,...,...
219,MPXV/Human/USA/CA-LACPHL-MA00547/2023,2023-06-12,OR500003,Garrigues et al.,Los Angeles County Public Health Laboratories,Los Angeles County
220,Monkeypox virus/Human/USA/CA-LACPHL-MA00689/2024,2024-08-30,PQ388214,Heibeck et al.,Los Angeles County Department of Public Health...,Los Angeles County
221,hMPX/Human/USA/CA-LACPHL-MA00061/2022,2022-07-20,OP440563,Hemarajata et al.,"Los Angeles County Public Health Laboratories,...",Los Angeles County
222,MPXV/Human/USA/CA-LACPHL-MA00336/2022,2022-11-09,OP890564,Hemarajata et al.,"Los Angeles County Public Health Laboratories,...",Los Angeles County


In [59]:
removed_meta["year"] = removed_meta['date'].str.rsplit("-", 0).str[-0].tolist()


In [61]:
removed_meta["year"].value_counts()

2022    131
2023     75
2024     18
Name: year, dtype: int64

In [71]:
def convert_format(number):
    date = dt.strptime(number, '%Y-%m-%d')
    date = date - timedelta(days=date.weekday())
    return date

In [72]:
removed_meta['year-week'] = removed_meta['date'].map(convert_format)


In [74]:
removed_meta["year-week"].value_counts()

2022-11-07    29
2022-10-03    19
2022-09-05    14
2022-07-25    10
2022-10-10    10
2023-05-01     8
2022-07-18     8
2023-04-24     8
2023-01-02     7
2022-10-24     7
2023-01-23     7
2023-04-10     6
2024-08-26     5
2022-10-17     5
2023-01-09     5
2022-09-26     5
2023-01-16     4
2022-11-21     3
2023-06-26     3
2023-02-13     3
2024-09-30     3
2023-06-05     3
2022-11-28     3
2022-08-15     3
2022-12-19     3
2023-02-20     3
2023-05-08     3
2024-01-22     2
2022-07-11     2
2022-12-05     2
2024-06-10     2
2023-06-12     2
2023-11-27     2
2022-08-08     2
2023-07-31     2
2022-09-12     2
2023-03-27     2
2022-12-26     1
2023-03-13     1
2024-09-09     1
2024-08-19     1
2024-01-15     1
2022-08-01     1
2023-10-23     1
2024-01-08     1
2024-06-17     1
2022-11-14     1
2023-08-07     1
2023-03-20     1
2024-12-09     1
2023-03-06     1
2022-12-12     1
2023-02-06     1
2023-08-14     1
Name: year-week, dtype: int64